In [26]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
df = pd.read_csv('/content/drive/MyDrive/watson_healthcare_modified.csv')
print(" Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

 Dataset loaded successfully!
Shape: (1676, 35)


,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,RelationshipSatisfaction,StandardHours,Shift,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1313919,41,No,Travel_Rarely,1102,Cardiology,1,2,Life Sciences,1,...,1,80,0,8,0,1,6,4,0,5
1,1200302,49,No,Travel_Frequently,279,Maternity,8,1,Life Sciences,1,...,4,80,1,10,3,3,10,7,1,7
2,1060315,37,Yes,Travel_Rarely,1373,Maternity,2,2,Other,1,...,2,80,0,7,3,3,0,0,0,0
3,1272912,33,No,Travel_Frequently,1392,Maternity,3,4,Life Sciences,1,...,3,80,0,8,3,3,8,7,3,0
4,1414939,27,No,Travel_Rarely,591,Maternity,2,1,Medical,1,...,4,80,1,6,3,3,2,2,2,2


In [29]:
employee_ids = df["EmployeeID"]

In [30]:
jobrole_groups = {
    'Nurse': 'Clinical', 'Doctor': 'Clinical', 'Surgeon': 'Clinical',
    'Lab Technician': 'Technical', 'Technician': 'Technical',
    'Healthcare Representative': 'Technical',
    'Manager': 'Administrative', 'HR': 'Administrative',
    'Clerk': 'Administrative', 'Sales Executive': 'Administrative',
    'Research Director': 'Administrative'
}

df['JobRoleGroup'] = df['JobRole'].map(jobrole_groups).fillna('Other')

df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})

In [31]:
selected_features = [
   'OverTime', 'JobRoleGroup', 'DistanceFromHome',
    'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance',
    'RelationshipSatisfaction', 'JobInvolvement', 'TrainingTimesLastYear',
    'YearsSinceLastPromotion', 'YearsInCurrentRole', 'TotalWorkingYears',
    'JobLevel', 'MonthlyIncome', 'PercentSalaryHike', 'PerformanceRating',
    'YearsWithCurrManager'
]
df = df[selected_features].copy()
print("\nSelected features for feature extraction:")
print(df.head())


Selected features for feature extraction:
   OverTime JobRoleGroup  DistanceFromHome  JobSatisfaction  \
0         1     Clinical                 1                4   
1         0        Other                 8                2   
2         1     Clinical                 2                3   
3         1        Other                 3                3   
4         0     Clinical                 2                2   

   EnvironmentSatisfaction  WorkLifeBalance  RelationshipSatisfaction  \
0                        2                1                         1   
1                        3                3                         4   
2                        4                3                         2   
3                        4                3                         3   
4                        1                3                         4   

   JobInvolvement  TrainingTimesLastYear  YearsSinceLastPromotion  \
0               3                      0                        0   
1

In [32]:
numeric_features = [
    'DistanceFromHome', 'JobSatisfaction', 'EnvironmentSatisfaction',
    'WorkLifeBalance', 'RelationshipSatisfaction', 'JobInvolvement',
    'TrainingTimesLastYear', 'YearsSinceLastPromotion', 'YearsInCurrentRole',
    'TotalWorkingYears', 'JobLevel', 'MonthlyIncome', 'PercentSalaryHike',
    'PerformanceRating', 'YearsWithCurrManager',
]
categorical_features = ['JobRoleGroup']

In [33]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),    # handle missing numbers
    ('scaler', MinMaxScaler())                        # scale to 0–1 range
])
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))  # one-hot encode text columns
])

feature_extraction_pipeline = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
])
remainder='passthrough'

print("\n Feature extraction pipeline created successfully!")


 Feature extraction pipeline created successfully!


In [34]:
# Apply pipeline
X_prepared = feature_extraction_pipeline.fit_transform(df)

# Get feature names
encoded_columns = feature_extraction_pipeline.get_feature_names_out().tolist()

# Rename remainder → OverTime
encoded_columns[-1] = 'OverTime'


clean_df = pd.DataFrame(X_prepared, columns=encoded_columns)


print(clean_df.head())
print("OverTime present? →", 'OverTime' in clean_df.columns)


   num__DistanceFromHome  num__JobSatisfaction  num__EnvironmentSatisfaction  \
0               0.000000              1.000000                      0.333333   
1               0.250000              0.333333                      0.666667   
2               0.035714              0.666667                      1.000000   
3               0.071429              0.666667                      1.000000   
4               0.035714              0.333333                      0.000000   

   num__WorkLifeBalance  num__RelationshipSatisfaction  num__JobInvolvement  \
0              0.000000                       0.000000             0.666667   
1              0.666667                       1.000000             0.333333   
2              0.666667                       0.333333             0.333333   
3              0.666667                       0.666667             0.666667   
4              0.666667                       1.000000             0.666667   

   num__TrainingTimesLastYear  num__YearsSin

In [35]:
clean_df["EmployeeID"] = employee_ids.values


In [36]:
output_path = '/content/clean_retention_features_updated_new1.csv'
clean_df.to_csv(output_path, index=False)
print(f"\n✅ Cleaned dataset saved to: {output_path}")
print("Shape:", df.shape)


✅ Cleaned dataset saved to: /content/clean_retention_features_updated_new1.csv
Shape: (1676, 17)


In [37]:
from google.colab import files
files.download('/content/clean_retention_features_updated_new1.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>